# Experiment 4 - Gaussian Noise and Gaussian Smoothing

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

In [ ]:
img = cv2.cvtColor(data.chelsea(), cv2.COLOR_RGB2GRAY)

MU, SIGMA = 0, 40
rng = np.random.default_rng(3)

noisy = np.clip(img + rng.normal(MU, SIGMA, img.shape), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, im, t in zip(axes, [img, noisy], ["original", f"gaussian noise sigma={SIGMA}"]):
    ax.imshow(im, cmap="gray")
    ax.set_title(t)
    ax.axis("off")
plt.show()

In [ ]:
KSIZE, KSIGMA = 11, 3

cv_blur = cv2.GaussianBlur(noisy, (KSIZE, KSIZE), KSIGMA)

plt.figure(figsize=(6, 5))
plt.imshow(cv_blur, cmap="gray")
plt.title(f"cv2.GaussianBlur {KSIZE}x{KSIZE}, sigma={KSIGMA}")
plt.axis("off")
plt.show()

In [ ]:
def gauss_kernel(size, sigma):
    r = size // 2
    t = np.arange(-r, r + 1, dtype=np.float64)
    g = np.exp(-(t ** 2) / (2 * sigma ** 2))
    g /= g.sum()
    return np.outer(g, g)

def gauss_blur(src, size=5, sigma=1.0):
    kern = gauss_kernel(size, sigma)
    r = size // 2
    padded = np.pad(src.astype(np.float64), r, mode="reflect")
    h, w = src.shape
    acc = np.zeros((h, w))
    for dy in range(size):
        for dx in range(size):
            acc += kern[dy, dx] * padded[dy:dy + h, dx:dx + w]
    return np.clip(np.rint(acc), 0, 255).astype(np.uint8)

k = gauss_kernel(5, 1.0)
print(np.round(k, 4))
print("sum:", round(float(k.sum()), 6))

In [ ]:
my_blur = gauss_blur(noisy, KSIZE, KSIGMA)

diff = np.abs(my_blur.astype(int) - cv_blur.astype(int))
print("mean abs diff vs opencv:", round(float(diff.mean()), 3))
print("max abs diff vs opencv :", int(diff.max()))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, im, t in zip(axes, [noisy, cv_blur, my_blur], ["noisy", "opencv", "custom"]):
    ax.imshow(im, cmap="gray")
    ax.set_title(t)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
sigmas = [1, 3, 6]
fig, axes = plt.subplots(1, len(sigmas), figsize=(16, 4))
for ax, s in zip(axes, sigmas):
    size = 2 * int(3 * s) + 1
    ax.imshow(gauss_blur(noisy, size, s), cmap="gray")
    ax.set_title(f"sigma={s}, k={size}")
    ax.axis("off")
plt.show()